# Batch SAM3 Auto-Labeling (text-promptable classes)

Labels the classes that passed the zero-shot gate (curbstone / speedbump / pole / ...)
across all their sequences. Per sequence:

1. camera from driving direction (both when unknown),
2. extract N mid-approach frames (cached - these become the training images),
3. **one** SAM3 text-prompt inference on a seed frame,
4. pick the target among distractors with an approach-corridor prior
   (horizontally central, below the horizon = "ahead of the bumper"),
5. propagate that instance's box with the SAM3 tracker across all frames,
6. append to `propagated_boxes.csv` - same schema as the click labeler, so
   `label_qa_review.ipynb` and the dataset merge consume it unchanged.

**Fully resumable**: labeled sequence/cameras are skipped on rerun; rows append after
every sequence; failures -> `failures.csv`; seed diagnostics -> `seed_log.csv`.

**Run a validation batch first** (`LIMIT_PER_OBJECT = 50`), QA it in
`label_qa_review.ipynb`, then set the limit to 0 for the full run
(~3,300 sequences at 10-20 s each on a T4 = 10-15 GPU-hours, safe across disconnects).

## 1. Setup

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

if IN_COLAB:
    pip_install("-U", "transformers", "accelerate")

import csv
import gc
import json
import random
import re
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"colab={IN_COLAB} device={DEVICE}")
if DEVICE != "cuda":
    print("WARNING: no GPU - labeling will be extremely slow")

## 2. Config

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def find_repo_root() -> Path | None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists():
            return base
    colab_clone = Path("/content/vision-uss-research")
    return colab_clone if colab_clone.exists() else None

PROFILE = "colab_drive" if IN_COLAB else "local"
RAW_ROOT_OVERRIDE = None
INVENTORY_CSV_OVERRIDE = None
OUT_DIR_OVERRIDE = None

repo_root = find_repo_root()
if repo_root:
    sys.path.insert(0, str(repo_root / "src"))
    from vision_uss_research.settings import load_paths
    paths = load_paths(profile=PROFILE)
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
    INVENTORY_CSV = Path(
        INVENTORY_CSV_OVERRIDE
        or paths.outputs_dir / "profiles" / "object_inventory" / "inventory_folders.csv")
    OUT_DIR = Path(OUT_DIR_OVERRIDE
                   or paths.outputs_dir / "experiments" / "sam3_text_labels")
else:
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE)
    INVENTORY_CSV = Path(INVENTORY_CSV_OVERRIDE)
    OUT_DIR = Path(OUT_DIR_OVERRIDE)

FRAMES_ROOT = OUT_DIR / "frames"
BOXES_CSV = OUT_DIR / "propagated_boxes.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- run knobs ---------------------------------------------------------------
OBJECTS = ["curbstone", "speedbump", "pole"]
LIMIT_PER_OBJECT = 50      # validation batch; 0 = all sequences
N_FRAMES = 20
RATIO_RANGE = (0.30, 0.85)
SEED_RATIO = 0.60
SCORE_THRESHOLD = 0.3
SEED = 0
DRY_RUN = False            # True: select + extract frames only, no model
SAM3_MODEL_ID = "facebook/sam3"

# Text prompts (appearance only - the class comes from sequence metadata).
OBJECT_PROMPTS = {
    "curbstone": "curb",
    "curbstone_side": "curb",
    "speedbump": "speed bump",
    "pole": "vertical pole",
    "squarepole": "vertical pole",
    "cone": "traffic cone",
    "bollard": "bollard",
    "car": "car",
    "tree": "tree trunk",
    "bush": "bush",
    "stonelarge": "large stone block",
    "stonemiddle": "stone block",
    "cubestandard": "cube-shaped test object",
    "ubarrier": "metal barrier",
    "fence": "fence",
}
unknown = [o for o in OBJECTS if o not in OBJECT_PROMPTS]
assert not unknown, f"no prompt defined for: {unknown}"

print(f"raw root  : {RAW_ROOT}")
print(f"inventory : {INVENTORY_CSV}")
print(f"output    : {OUT_DIR}")

## 3. Helpers (camera selection, frame extraction, corridor picker)

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".webm", ".avi"}
BOX_COLS = ["sequence_id", "target_object", "camera", "frame_idx", "frame_path",
            "x0", "y0", "x1", "y1", "mask_area_frac"]

def with_drive_retry(fn, *args, retries=2, wait_s=3):
    for attempt in range(retries + 1):
        try:
            return fn(*args)
        except OSError:
            if attempt == retries:
                raise
            time.sleep(wait_s)

def sequence_files(seq_dir: Path) -> dict:
    top = [p for p in seq_dir.iterdir() if p.is_file()]
    def pick(pred):
        matches = sorted(p for p in top if pred(p))
        return matches[0] if matches else None
    return {
        "metadata_json": pick(lambda p: p.suffix.lower() == ".json"),
        "front_video": pick(lambda p: "front" in p.name.lower()
                            and p.suffix.lower() in VIDEO_EXTENSIONS),
        "rear_video": pick(lambda p: "rear" in p.name.lower()
                           and p.suffix.lower() in VIDEO_EXTENSIONS),
    }

def select_camera_videos(seq_dir: Path) -> list[tuple[str, Path]]:
    found = sequence_files(seq_dir)
    direction = "unknown"
    if found["metadata_json"]:
        with found["metadata_json"].open() as f:
            metadata = json.load(f)
        for tag in metadata.get("tags", []):
            if tag.get("tagSubCategory", {}).get("name", "").lower() == "driving direction":
                direction = tag.get("name", "unknown")
    if direction == "backward":
        videos = [("rear", found["rear_video"])]
    elif direction == "forward":
        videos = [("front", found["front_video"])]
    else:
        videos = [("front", found["front_video"]), ("rear", found["rear_video"])]
    return [(c, p) for c, p in videos if p is not None]

def extract_sequence_frames(seq_id: str, camera: str, video_path: Path) -> Path | None:
    out_dir = FRAMES_ROOT / seq_id / camera
    if out_dir.exists() and len(list(out_dir.glob("*.jpg"))) >= N_FRAMES - 2:
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    duration_ms = 1000.0 * cap.get(cv2.CAP_PROP_FRAME_COUNT) / max(cap.get(cv2.CAP_PROP_FPS), 1e-6)
    ratios = np.linspace(RATIO_RANGE[0], RATIO_RANGE[1], N_FRAMES)
    kept = 0
    for i, r in enumerate(ratios):
        cap.set(cv2.CAP_PROP_POS_MSEC, r * duration_ms)
        ok, frame = cap.read()
        if ok:
            cv2.imwrite(str(out_dir / f"{i:05d}.jpg"), frame,
                        [cv2.IMWRITE_JPEG_QUALITY, 92])
            kept += 1
    cap.release()
    return out_dir if kept else None

def corridor_weight(cx_norm: float, cy_norm: float,
                    sigma: float = 0.25, min_cy: float = 0.35) -> float:
    """Prior for 'the staged target ahead of the bumper': horizontally central,
    below the horizon band."""
    horizontal = float(np.exp(-((cx_norm - 0.5) ** 2) / (2 * sigma ** 2)))
    vertical = 1.0 if cy_norm >= min_cy else 0.25
    return horizontal * vertical

def pick_target_instance(result: dict, image_shape: tuple) -> tuple:
    h, w = image_shape
    n = len(result["scores"])
    if n == 0:
        return None, {"n_candidates": 0}
    combined = []
    for i in range(n):
        x0, y0, x1, y1 = result["boxes"][i]
        cx_norm, cy_norm = ((x0 + x1) / 2) / w, ((y0 + y1) / 2) / h
        combined.append(float(result["scores"][i]) * corridor_weight(cx_norm, cy_norm))
    best = int(np.argmax(combined))
    return best, {"n_candidates": n,
                  "seed_score": round(float(result["scores"][best]), 3),
                  "combined_score": round(combined[best], 3)}

def masks_to_bbox(mask: np.ndarray):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def append_csv(path: Path, rows: list, cols: list) -> None:
    write_header = not path.exists()
    with path.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=cols)
        if write_header:
            writer.writeheader()
        writer.writerows(rows)

def free_gpu():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 4. Build the work list from the inventory (resume-aware)

In [ ]:
inv = pd.read_csv(INVENTORY_CSV)
usable = inv[(inv["status"] == "complete")
             & ((inv["has_front_video"] == 1) | (inv["has_rear_video"] == 1))]

rng = random.Random(SEED)
work = []
for obj in OBJECTS:
    mask = usable["label_v2_objects"].fillna("").str.split("|").apply(
        lambda names: any(n.strip().lower() == obj for n in names))
    candidates = sorted(usable[mask]["sequence_id"].tolist())
    if LIMIT_PER_OBJECT:
        candidates = sorted(rng.sample(candidates,
                                       min(LIMIT_PER_OBJECT, len(candidates))))
    work += [{"sequence_id": s, "target_object": obj} for s in candidates]
    print(f"{obj:<15} {len(candidates)} sequences queued")

done_keys: set[str] = set()
if BOXES_CSV.exists():
    prev = pd.read_csv(BOXES_CSV)
    done_keys = set(prev["sequence_id"].astype(str) + "::" + prev["camera"].astype(str))
    print(f"\nresuming: {len(done_keys)} sequence/cameras already labeled - will skip")
print(f"total queued: {len(work)} sequences")

## 5. Load SAM3 (text prompting + tracker, one checkpoint)

If your transformers version exposes different names, check
`https://huggingface.co/facebook/sam3` and adjust the API-marked lines.

In [ ]:
if not DRY_RUN:
    from transformers import (Sam3Model, Sam3Processor,
                              Sam3TrackerVideoModel, Sam3TrackerVideoProcessor)

    DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
    pcs_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID)
    pcs_model = Sam3Model.from_pretrained(SAM3_MODEL_ID).to(DEVICE).eval()
    tracker = (Sam3TrackerVideoModel.from_pretrained(SAM3_MODEL_ID)
               .to(DEVICE, dtype=DTYPE).eval())
    tracker_processor = Sam3TrackerVideoProcessor.from_pretrained(SAM3_MODEL_ID)

@torch.inference_mode()
def segment_text(image_rgb: np.ndarray, text_prompt: str) -> dict:
    pil = Image.fromarray(image_rgb)
    inputs = pcs_processor(images=pil, text=text_prompt,
                           return_tensors="pt").to(DEVICE)                # API line
    outputs = pcs_model(**inputs)
    results = pcs_processor.post_process_instance_segmentation(          # API line
        outputs, threshold=SCORE_THRESHOLD, mask_threshold=0.5,
        target_sizes=[(pil.height, pil.width)])[0]
    return {"masks": results["masks"].cpu().numpy().astype(bool),
            "boxes": results["boxes"].cpu().numpy(),
            "scores": results["scores"].cpu().numpy()}

@torch.inference_mode()
def track_forward(frames: list, seed_idx: int, seed_box: tuple) -> dict:
    h, w = frames[0].height, frames[0].width
    session = tracker_processor.init_video_session(                       # API line
        video=frames, inference_device=DEVICE, dtype=DTYPE)
    tracker_processor.add_inputs_to_inference_session(                    # API line
        session, frame_idx=seed_idx, obj_ids=1,
        input_boxes=[[list(map(float, seed_box))]])
    masks = {}
    for output in tracker.propagate_in_video_iterator(                    # API line
            session, start_frame_idx=seed_idx):
        video_masks = tracker_processor.post_process_masks(
            [output.pred_masks], original_sizes=[[h, w]], binarize=True)[0]
        masks[output.frame_idx] = video_masks[0, 0].cpu().numpy().astype(bool)
    return masks

def propagate(frames_dir: Path, seed_idx: int, seed_box: tuple) -> dict:
    frames = [Image.open(p).convert("RGB") for p in sorted(frames_dir.glob("*.jpg"))]
    results = track_forward(frames, seed_idx, seed_box)
    free_gpu()
    if seed_idx > 0:
        rev = track_forward(list(reversed(frames)),
                            len(frames) - 1 - seed_idx, seed_box)
        for ridx, mask in rev.items():
            results.setdefault(len(frames) - 1 - ridx, mask)
    free_gpu()
    return results

## 6. Main labeling loop (resumable - rerun this cell any time)

In [ ]:
SEED_IDX = int(round((SEED_RATIO - RATIO_RANGE[0])
                     / (RATIO_RANGE[1] - RATIO_RANGE[0]) * (N_FRAMES - 1)))
stats = defaultdict(int)

for item in tqdm(work, desc="Labeling sequences", unit="seq"):
    seq_id, obj = item["sequence_id"], item["target_object"]
    seq_dir = RAW_ROOT / seq_id
    prompt = OBJECT_PROMPTS[obj]
    try:
        videos = with_drive_retry(select_camera_videos, seq_dir)
    except Exception as e:
        append_csv(OUT_DIR / "failures.csv",
                   [{"sequence_id": seq_id, "camera": "", "stage": "select_camera",
                     "error": f"{type(e).__name__}: {e}"}],
                   ["sequence_id", "camera", "stage", "error"])
        stats["failed"] += 1
        continue

    for camera, video_path in videos:
        key = f"{seq_id}::{camera}"
        if key in done_keys:
            stats["skipped_done"] += 1
            continue
        try:
            frames_dir = with_drive_retry(extract_sequence_frames,
                                          seq_id, camera, video_path)
            if frames_dir is None:
                raise RuntimeError("no frames extracted")
            seed_path = frames_dir / f"{SEED_IDX:05d}.jpg"
            if not seed_path.exists():
                available = sorted(frames_dir.glob("*.jpg"))
                if not available:
                    raise RuntimeError("empty frames dir")
                seed_path = min(available, key=lambda p: abs(int(p.stem) - SEED_IDX))
            actual_seed_idx = int(seed_path.stem)

            if DRY_RUN:
                stats["dry_ok"] += 1
                continue

            image_rgb = cv2.cvtColor(cv2.imread(str(seed_path)), cv2.COLOR_BGR2RGB)
            result = None
            for attempt in range(2):
                try:
                    result = segment_text(image_rgb, prompt)
                    break
                except torch.cuda.OutOfMemoryError:
                    print(f"OOM (seed) {seq_id} - clearing cache, retry {attempt + 1}/2")
                    free_gpu()
            if result is None:
                raise RuntimeError("seed segmentation failed after OOM retries")

            best, diag = pick_target_instance(result, image_rgb.shape[:2])
            append_csv(OUT_DIR / "seed_log.csv",
                       [{"sequence_id": seq_id, "camera": camera, "target_object": obj,
                         "seed_frame_idx": actual_seed_idx, **diag}],
                       ["sequence_id", "camera", "target_object", "seed_frame_idx",
                        "n_candidates", "seed_score", "combined_score"])
            if best is None:
                stats["no_seed_detection"] += 1
                continue
            seed_box = tuple(map(float, result["boxes"][best]))

            masks = None
            for attempt in range(2):
                try:
                    masks = propagate(frames_dir, actual_seed_idx, seed_box)
                    break
                except torch.cuda.OutOfMemoryError:
                    print(f"OOM (track) {seq_id} - clearing cache, retry {attempt + 1}/2")
                    free_gpu()
            if masks is None:
                raise RuntimeError("propagation failed after OOM retries")

            rows = []
            for frame_idx, mask in sorted(masks.items()):
                bbox = masks_to_bbox(mask)
                rows.append({
                    "sequence_id": seq_id, "target_object": obj, "camera": camera,
                    "frame_idx": frame_idx,
                    "frame_path": str(frames_dir / f"{frame_idx:05d}.jpg"),
                    "x0": bbox[0] if bbox else None,
                    "y0": bbox[1] if bbox else None,
                    "x1": bbox[2] if bbox else None,
                    "y1": bbox[3] if bbox else None,
                    "mask_area_frac": float(mask.mean()),
                })
            append_csv(BOXES_CSV, rows, BOX_COLS)
            done_keys.add(key)
            stats["labeled"] += 1
        except Exception as e:
            free_gpu()
            append_csv(OUT_DIR / "failures.csv",
                       [{"sequence_id": seq_id, "camera": camera, "stage": "label",
                         "error": f"{type(e).__name__}: {e}"}],
                       ["sequence_id", "camera", "stage", "error"])
            stats["failed"] += 1

print("\nsummary:")
for k, v in sorted(stats.items()):
    print(f"  {k:<18} {v}")
print(f"\nboxes    : {BOXES_CSV}")
print(f"seed log : {OUT_DIR / 'seed_log.csv'}")
print(f"failures : {OUT_DIR / 'failures.csv'}")

## 7. Quick health check on this run

In [ ]:
if BOXES_CSV.exists():
    boxes = pd.read_csv(BOXES_CSV)
    health = (boxes.assign(has_box=boxes["x0"].notna())
              .groupby("target_object")
              .agg(sequences=("sequence_id", "nunique"),
                   frames=("has_box", "size"),
                   frames_with_box=("has_box", "sum"),
                   area_frac_median=("mask_area_frac", "median")))
    health["coverage"] = (health["frames_with_box"] / health["frames"]).round(3)
    display(health.round(4))

if (OUT_DIR / "seed_log.csv").exists():
    seed_log = pd.read_csv(OUT_DIR / "seed_log.csv")
    print("seed candidates per object (distractor pressure):")
    display(seed_log.groupby("target_object")["n_candidates"]
            .describe()[["count", "mean", "50%", "max"]].round(2))

## Next steps

1. Add this run to `LABEL_SOURCES` in `notebooks/label_qa_review.ipynb`:
   ```python
   {"name": "sam3_text",
    "boxes_csv": OUTPUTS_ROOT / "experiments" / "sam3_text_labels" / "propagated_boxes.csv",
    "frames_root": OUTPUTS_ROOT / "experiments" / "sam3_text_labels" / "frames"}
   ```
   and check galleries + flags, especially whether the corridor picker chose the
   *labeled* curb rather than a background one.
2. If the validation batch looks right: set `LIMIT_PER_OBJECT = 0` and rerun cells
   4 and 6 for the full run (everything already labeled is skipped).
3. Wrong-instance picks -> tune `corridor_weight` (sigma / min_cy) or
   `SCORE_THRESHOLD`, delete the affected rows, rerun.
4. After QA passes: dataset merge -> sequence-disjoint splits -> YOLO training
   (hold out ~20% of the *clicked* sequences as the gold test set).